# Assignment 1 - Srijat Verma

In [1]:
import psycopg2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [2]:
directory = "T6-gsa/Session_5-6/"

We first establish a connection to our database

In [3]:
conn = psycopg2.connect(dbname="postgis", 
                 user="gsa2022", 
                 password="g5!V%T1Vmd", 
                 host="192.168.212.99", 
                 port=32771)

In [4]:
# Check the available schema in our database:
pd.read_sql('''
            SELECT nspname
            FROM pg_catalog.pg_namespace
            ''', conn).values

array([['pg_toast'],
       ['pg_temp_1'],
       ['pg_toast_temp_1'],
       ['pg_catalog'],
       ['br_20000801'],
       ['information_schema'],
       ['topology'],
       ['br_20100801'],
       ['public'],
       ['gadm'],
       ['ph_20200501'],
       ['pg_temp_7'],
       ['pg_toast_temp_7'],
       ['pg_temp_8'],
       ['govt'],
       ['namria_hazard'],
       ['pg_toast_temp_8'],
       ['pg_temp_15'],
       ['pedestrians'],
       ['pg_temp_4'],
       ['pg_toast_temp_15'],
       ['pg_toast_temp_4'],
       ['ph_20150801'],
       ['pg_temp_21'],
       ['deped'],
       ['pg_toast_temp_21'],
       ['au_20110809'],
       ['dpwh'],
       ['pg_temp_14'],
       ['au_20160809'],
       ['pg_toast_temp_14'],
       ['pg_temp_45'],
       ['pg_toast_temp_45'],
       ['gadm_v4_1'],
       ['au_20210810'],
       ['bgd_20010127'],
       ['bgd_20110319'],
       ['bgd_20220621'],
       ['br_20220731'],
       ['in_20000630'],
       ['in_20100630'],
       ['in_20200930'

We will be using two schemas:
1. Public : contains OSM data
2. GADM : contains GADM data

In [5]:
#Check the contents of the public schema
pd.read_sql('''
            SELECT *
            FROM information_schema.tables
            WHERE table_schema = 'public'
            ''', conn)

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action
0,postgis,public,spatial_ref_sys,BASE TABLE,None,None,None,None,None,YES,NO,None
1,postgis,public,ph_line,BASE TABLE,None,None,None,None,None,YES,NO,None
2,postgis,public,pop_census_place,BASE TABLE,None,None,None,None,None,YES,NO,None
3,postgis,public,geography_columns,VIEW,None,None,None,None,None,NO,NO,None
4,postgis,public,ph_roads,BASE TABLE,None,None,None,None,None,YES,NO,None
5,postgis,public,au_point,BASE TABLE,None,None,None,None,None,YES,NO,None
6,postgis,public,geometry_columns,VIEW,None,None,None,None,None,YES,NO,None
7,postgis,public,ph_polygon,BASE TABLE,None,None,None,None,None,YES,NO,None
8,postgis,public,ph_point,BASE TABLE,None,None,None,None,None,YES,NO,None
9,postgis,public,au_roads,BASE TABLE,None,None,None,None,None,YES,NO,None


In [6]:
#Check the contents of the GADM schema
pd.read_sql('''
            SELECT *
            FROM information_schema.tables
            WHERE table_schema = 'gadm'
            ''', conn)

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action
0,postgis,gadm,adm0,BASE TABLE,None,None,None,None,None,YES,NO,None
1,postgis,gadm,adm1,BASE TABLE,None,None,None,None,None,YES,NO,None
2,postgis,gadm,adm4,BASE TABLE,None,None,None,None,None,YES,NO,None
3,postgis,gadm,adm3,BASE TABLE,None,None,None,None,None,YES,NO,None
4,postgis,gadm,adm5,BASE TABLE,None,None,None,None,None,YES,NO,None
5,postgis,gadm,ph_brgy,BASE TABLE,None,None,None,None,None,YES,NO,None
6,postgis,gadm,ph_hum,BASE TABLE,None,None,None,None,None,YES,NO,None
7,postgis,gadm,adm2,BASE TABLE,None,None,None,None,None,YES,NO,None
8,postgis,gadm,ph,BASE TABLE,None,None,None,None,None,YES,NO,None
9,postgis,gadm,au,BASE TABLE,None,None,None,None,None,YES,NO,None


In [7]:
import geopandas as gpd

## Assignment

Suppose that you are the *head of branch expansion* of food chain "McDollibee" with the base of operations located here at AIM. You are tasked to create a strategy that considers the following:
1. Level of Urbanization - You want to expand to a location where the amenity density is in top 10 percentile. 
2. Market Availability and Competition - You cater to students but you prefer areas with limited competition
3. Logistics - The farther the location is from AIM, the more expensive building the branch will be.

Using the information available to you, create a report on what possible locations fit the requirements of your branch expansion strategy.

In [8]:
AIM = 'POINT(121.0244 14.5573)'

In [9]:
#Your Code and Report Here
pd.read_sql('''
    WITH amenity_counts AS (
        SELECT g.name_2,
               g.geom,
               COUNT(p.way) as amenity_count,
               COUNT(p.way) / (st_area(st_transform(g.geom, 3123)) / 1000000) as amenity_density,
               NTILE(10) OVER (ORDER BY COUNT(p.way) / (st_area(st_transform(g.geom, 3123)) / 1000000)) as percentile
        FROM gadm.ph g
        JOIN public.ph_point p ON st_within(p.way, g.geom)
        WHERE p.amenity IS NOT NULL
        GROUP BY g.name_2, g.geom
    )
    SELECT name_2, amenity_count, amenity_density
    FROM amenity_counts
    WHERE percentile = 10
    ORDER BY amenity_density DESC
''', conn)

,name_2,amenity_count,amenity_density
0,Makati City,2333,73.481489
1,San Juan,406,70.239652
2,Manila,1863,50.806953
3,Mandaluyong,492,44.968149
4,Pateros,75,38.363473
...,...,...,...
131,Santo Tomas,13,0.928125
132,Ajuy,156,0.917719
133,Bangar,40,0.887451
134,Ozamis City,131,0.874131


In [10]:
pd.read_sql('''
    SELECT g.name_2,
           COUNT(CASE WHEN p.amenity = 'university' OR p.amenity = 'school' THEN 1 END) as student_anchors,
           COUNT(CASE WHEN p.amenity = 'fast_food' THEN 1 END) as competitor_count,
           COUNT(CASE WHEN p.amenity = 'university' OR p.amenity = 'school' THEN 1 END) * 1.0 /
           NULLIF(COUNT(CASE WHEN p.amenity = 'fast_food' THEN 1 END), 0) as opportunity_score
    FROM gadm.ph g
    JOIN public.ph_point p ON st_within(p.way, g.geom)
    WHERE p.amenity IS NOT NULL
    GROUP BY g.name_2
    ORDER BY opportunity_score DESC
    LIMIT 20
''', conn)

,name_2,student_anchors,competitor_count,opportunity_score
0,Alcantara,2,0,None
1,Albuera,0,0,None
2,Alcala,3,0,None
3,Aborlan,0,0,None
4,Abucay,3,0,None
5,Agdangan,3,0,None
6,Aglipay,1,0,None
7,Albuquerque,1,0,None
8,Agoncillo,1,0,None
9,Abuyog,1,0,None


In [11]:
pd.read_sql('''
    SELECT g.name_2,
           st_distance(
               st_transform(st_centroid(g.geom), 3123),
               st_transform(ST_GeomFromText(\'POINT(121.0244 14.5573)\', 4326), 3123)
           ) / 1000 as distance_from_AIM_km
    FROM gadm.ph g
    ORDER BY distance_from_AIM_km ASC
    LIMIT 20
''', conn)

,name_2,distance_from_aim_km
0,Makati City,1.318673
1,Mandaluyong,3.371704
2,Pasay City,3.630146
3,Pateros,4.978349
4,San Juan,5.152216
5,Manila,6.353297
6,Taguig,6.509191
7,Pasig City,7.025454
8,Parañaque,8.195754
9,Cainta,10.621045


In [12]:
pd.read_sql('''
    WITH amenity_stats AS (
        SELECT g.name_2,
               COUNT(CASE WHEN p.amenity IN (\'university\', \'school\') THEN 1 END) as student_anchors,
               COUNT(CASE WHEN p.amenity = \'fast_food\' THEN 1 END) as competitors,
               NTILE(10) OVER (ORDER BY COUNT(p.way) / (st_area(st_transform(g.geom, 3123)) / 1000000)) as density_percentile
        FROM gadm.ph g
        JOIN public.ph_point p ON st_within(p.way, g.geom)
        WHERE p.amenity IS NOT NULL
        GROUP BY g.name_2, g.geom
    )
    SELECT name_2, student_anchors, competitors
    FROM amenity_stats
    WHERE density_percentile = 10
''', conn)

,name_2,student_anchors,competitors
0,Indang,21,3
1,Ozamis City,20,6
2,Bangar,20,0
3,Ajuy,26,0
4,Santo Tomas,3,0
...,...,...,...
131,Pateros,9,12
132,Mandaluyong,9,83
133,Manila,118,322
134,San Juan,7,32


In [13]:
pd.read_sql('''
    WITH amenity_stats AS (
        SELECT g.name_2,
               g.geom,
               COUNT(p.way) / (st_area(st_transform(g.geom, 3123)) / 1000000) as amenity_density,
               COUNT(CASE WHEN p.amenity IN (\'university\', \'school\') THEN 1 END) as student_anchors,
               COUNT(CASE WHEN p.amenity = \'restaurant\' THEN 1 END) as competitors,
               NTILE(10) OVER (ORDER BY COUNT(p.way) / (st_area(st_transform(g.geom, 3123)) / 1000000)) as density_percentile,
               ROUND((st_distance(
                   st_transform(st_centroid(g.geom), 3123),
                   st_transform(ST_GeomFromText(\'POINT(121.0244 14.5573)\', 4326), 3123)
               ) / 1000)::numeric, 2) as distance_from_AIM_km
        FROM gadm.ph g
        JOIN public.ph_point p ON st_within(p.way, g.geom)
        WHERE p.amenity IS NOT NULL
        GROUP BY g.name_2, g.geom
    )
    SELECT name_2,
           amenity_density,
           student_anchors,
           competitors,
           distance_from_AIM_km,
           ROUND((student_anchors * 1.0 / (competitors + 1))::numeric, 2) as opportunity_score
    FROM amenity_stats
    WHERE density_percentile = 10
    AND distance_from_AIM_km <= 10
    ORDER BY opportunity_score DESC, distance_from_AIM_km ASC
    LIMIT 10
''', conn)

,name_2,amenity_density,student_anchors,competitors,distance_from_aim_km,opportunity_score
0,Pateros,38.363473,9,8,4.98,1.00
1,Taguig,7.956711,15,16,6.51,0.88
2,Manila,50.806953,118,263,6.35,0.45
3,Parañaque,19.112097,40,166,8.20,0.24
4,Pasig City,34.170640,49,218,7.03,0.22
5,Mandaluyong,44.968149,9,82,3.37,0.11
6,Pasay City,38.084522,14,150,3.63,0.09
7,Makati City,73.481489,40,678,1.32,0.06
8,San Juan,70.239652,7,118,5.15,0.06


### Primary Recommendation: Pateros
Pateros ranks first across all three evaluation criteria within the feasible logistics range:
- Urbanization: Amenity density of 38.36 per sq km, firmly in the top 10th percentile.
- Market Opportunity: Highest opportunity score of 1.00 — 9 student anchors against only 8 competitors, the best student-to-competitor ratio among all candidates.
- Logistics: 4.98 km from AIM — the closest qualifying city outside the already saturated Makati and Mandaluyong areas.
Pateros is a small but densely developed municipality with low existing restaurant competition, making it an ideal first-mover opportunity for McDollibee.

### Secondary Recommendation: Taguig
Taguig presents a strong secondary option with a good balance of urban development and market opportunity:
- Urbanization: Amenity density of 7.96 per sq km, qualifying in the top percentile.
- Market Opportunity: 15 student anchors and only 16 competitors — opportunity score of 0.88, second highest among all candidates.
- Logistics: 6.51 km from AIM, well within supply chain range.
Taguig is a rapidly growing city with significant residential and commercial development, offering long-term growth potential beyond the initial student demographic.

Based on the three-factor geospatial analysis, Pateros emerges as the most strategically sound location for McDollibee's first branch expansion. It satisfies all three requirements: it sits within the top urbanization percentile, offers the best student-to-competitor ratio among nearby cities, and remains within a cost-effective logistics range from AIM.
Taguig is recommended as a strong backup or simultaneous second expansion target, given its rapidly developing urban landscape and reasonable competitive environment.